In [ ]:
import os
import psycopg2
import pandas as pd
from google.cloud import bigquery
from psycopg2.extras import RealDictCursor
from dotenv import load_dotenv

load_dotenv()
#get table name and data from PG

conn = psycopg2.connect(
            host=os.environ["SOURCE_DB_HOST"],
            port=os.environ["SOURCE_DB_PORT"],
            dbname=os.environ["SOURCE_DB_NAME"],
            user=os.environ["SOURCE_DB_USER"],
            password=os.environ["SOURCE_DB_PASSWORD"],
    )
data = {}
try:
    with conn.cursor(cursor_factory=RealDictCursor) as cur:
        # BƯỚC 1: Câu lệnh SQL đặc biệt để lấy danh sách TÊN TẤT CẢ CÁC BẢNG do bạn tạo ra (schema 'public')
        cur.execute("""
            SELECT table_name 
            FROM information_schema.tables 
            WHERE table_schema = 'public' AND table_type = 'BASE TABLE';
        """)
        all_tables = [row['table_name'] for row in cur.fetchall()]
        print(all_tables)

        
        for table in all_tables:
            print(f"🔄 Đang lấy dữ liệu từ bảng: {table}...")
            cur.execute(f"SELECT * FROM {table}")
            rows = [dict(r) for r in cur.fetchall()]
            data[table] = rows
            print(f"✅ Đã lấy xong {len(rows)} dòng từ bảng {table}.\n")


        #get table schema from PG: 
        cur.execute("""
                SELECT table_name, column_name, data_type
                FROM information_schema.columns
                WHERE table_schema = 'public'
                ORDER BY table_name, ordinal_position
            """)
        schemas = {}
        for r in cur.fetchall():
            schemas.setdefault(r["table_name"], []).append(r)
finally:
    conn.close()

['transaction_status_history', 'cards', 'accounts', 'merchants', 'currencies', 'transaction_types', 'transactions', 'channels', 'customers', 'branches', 'products']
🔄 Đang lấy dữ liệu từ bảng: transaction_status_history...
✅ Đã lấy xong 130126 dòng từ bảng transaction_status_history.

🔄 Đang lấy dữ liệu từ bảng: cards...
✅ Đã lấy xong 4000 dòng từ bảng cards.

🔄 Đang lấy dữ liệu từ bảng: accounts...
✅ Đã lấy xong 7000 dòng từ bảng accounts.

🔄 Đang lấy dữ liệu từ bảng: merchants...
✅ Đã lấy xong 500 dòng từ bảng merchants.

🔄 Đang lấy dữ liệu từ bảng: currencies...
✅ Đã lấy xong 3 dòng từ bảng currencies.

🔄 Đang lấy dữ liệu từ bảng: transaction_types...
✅ Đã lấy xong 5 dòng từ bảng transaction_types.

🔄 Đang lấy dữ liệu từ bảng: transactions...
✅ Đã lấy xong 50000 dòng từ bảng transactions.

🔄 Đang lấy dữ liệu từ bảng: channels...
✅ Đã lấy xong 5 dòng từ bảng channels.

🔄 Đang lấy dữ liệu từ bảng: customers...
✅ Đã lấy xong 5000 dòng từ bảng customers.

🔄 Đang lấy dữ liệu từ bảng: bra

In [ ]:
PG_TO_BQ = {
    "smallint": "INT64",
    "integer": "INT64",
    "bigint": "INT64",
    "real": "FLOAT64",
    "double precision": "FLOAT64",
    "numeric": "NUMERIC",
    "boolean": "BOOL",
    "character varying": "STRING",
    "character": "STRING",
    "text": "STRING",
    "uuid": "STRING",
    "date": "DATE",
    "timestamp without time zone": "DATETIME",
    "timestamp with time zone": "TIMESTAMP",
    "json": "JSON",
    "jsonb": "JSON",
}

In [ ]:
import json
client = bigquery.Client()
dataset = os.getenv("BQ_DATASET")
for table_name, rows in data.items():
    bq_schema = []
    if table_name in schemas:
        for col in schemas[table_name]:
            col_name = col['column_name']
            bq_type = PG_TO_BQ.get(col['data_type'],"STRING")
            bq_schema.append(bigquery.SchemaField(col_name, bq_type, mode="NULLABLE"))
    
    # Thiết lập cấu hình Load Job với Schema cụ thể
    table_id = f"{client.project}.{dataset}.{table_name}"
    job_config = bigquery.LoadJobConfig(
        write_disposition="WRITE_TRUNCATE",
        schema=bq_schema)
    payload = json.loads(json.dumps(rows, default=str))
    print(f"Đang nạp {len(df)} dòng vào bảng {table_id}...")
    job = client.load_table_from_json(payload, table_id, job_config=job_config)
    job.result()  # Chờ job hoàn thành
    
    print(f"[load] Thành công: nạp {job.output_rows} dòng vào {table_id} WRITE_TRUNCATE")



Đang nạp 130126 dòng vào bảng project-9e3c74f0-c606-4810-b1c.core_banking_bronze.transaction_status_history...
[load] Thành công: nạp 130126 dòng vào project-9e3c74f0-c606-4810-b1c.core_banking_bronze.transaction_status_history WRITE_TRUNCATE
Đang nạp 130126 dòng vào bảng project-9e3c74f0-c606-4810-b1c.core_banking_bronze.cards...
[load] Thành công: nạp 130126 dòng vào project-9e3c74f0-c606-4810-b1c.core_banking_bronze.cards WRITE_TRUNCATE
Đang nạp 130126 dòng vào bảng project-9e3c74f0-c606-4810-b1c.core_banking_bronze.accounts...
[load] Thành công: nạp 130126 dòng vào project-9e3c74f0-c606-4810-b1c.core_banking_bronze.accounts WRITE_TRUNCATE
Đang nạp 130126 dòng vào bảng project-9e3c74f0-c606-4810-b1c.core_banking_bronze.merchants...
[load] Thành công: nạp 130126 dòng vào project-9e3c74f0-c606-4810-b1c.core_banking_bronze.merchants WRITE_TRUNCATE
Đang nạp 130126 dòng vào bảng project-9e3c74f0-c606-4810-b1c.core_banking_bronze.currencies...
[load] Thành công: nạp 130126 dòng vào projec